# APIM ❤️ AI Agents

## Simplified API to MCP lab
![flow](../../images/model-context-protocol.gif)

This focused lab shows the minimum setup to expose one mocked Weather REST API through the [Model Context Protocol](https://modelcontextprotocol.io/) with Azure API Management.

Follow this [guide](./COPILOT-STUDIO.md) to use the Weather MCP server in Copilot Studio.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`.


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according to your preferences.
- This simplified lab keeps the Weather API and Weather MCP server only.


In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

subscription_id = utils.get_current_subscription()
deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"  # change the name to match your naming style or an existing resource group
resource_group_location = "ukwest"

apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

apic_location = "uksouth"
apic_service_name_prefix = 'apic6'

utils.print_ok('Notebook initialized')


⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:58:19.323606 :1s]
👉🏽 Using Subscription ID: 784e6c68-d702-4a8b-a678-2e0f2f36deab (Visual Studio Enterprise Subscription)
✅ Notebook initialized ⌚ 14:58:19.323606 


<a id='1'></a>
### 1️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to define the resources declaratively. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.


In [2]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)


⚙️ Running: az group show --name lab-simplified-api-mcp 
👉🏽 Resource group lab-simplified-api-mcp does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name lab-simplified-api-mcp --location ukwest --tags source=ai-gateway 
✅ Resource group 'lab-simplified-api-mcp' created ⌚ 14:58:55.081210 :10s]


In [3]:
# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "apicLocation": {"value": apic_location},
        "apicServiceNamePrefix": {"value": apic_service_name_prefix}
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))


In [4]:
# Run the deployment
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)


⚙️ Running: az deployment group create --name simplified-api-mcp --resource-group lab-simplified-api-mcp --template-file main.bicep --parameters params.json 
✅ Deployment 'simplified-api-mcp' succeeded ⌚ 15:03:40.959134 :7s]


<a id='2'></a>
### 2️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.


In [5]:
# Obtain all of the outputs from the deployment
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    weather_api_endpoint = utils.get_deployment_output(output, 'weatherApiEndpoint', 'Weather API Endpoint')
    weather_mcp_endpoint = utils.get_deployment_output(output, 'weatherMcpEndpoint', 'Weather MCP Endpoint')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    utils.print_info('The Weather API and Weather MCP endpoints in this lab do not require authentication.')


⚙️ Running: az deployment group show --name simplified-api-mcp -g lab-simplified-api-mcp 
✅ Retrieved deployment: simplified-api-mcp ⌚ 15:04:38.973862 :4s]
👉🏽 Log Analytics Id: b8c10ff7-efdb-465a-9f13-974afaa65696
👉🏽 APIM Service Id: /subscriptions/784e6c68-d702-4a8b-a678-2e0f2f36deab/resourceGroups/lab-simplified-api-mcp/providers/Microsoft.ApiManagement/service/apim-quox5nktyqj66
👉🏽 APIM API Gateway URL: https://apim-quox5nktyqj66.azure-api.net
👉🏽 Weather API Endpoint: https://apim-quox5nktyqj66.azure-api.net/weather
👉🏽 Weather MCP Endpoint: https://apim-quox5nktyqj66.azure-api.net/weather-mcp/mcp
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****76d7
👉🏽 The Weather API and Weather MCP endpoints in this lab do not require authentication.


<a id='testconnection'></a>
### 🧪 Test the connection to the Weather MCP server and list tools



In [ ]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def list_tools(server_url):
    async with streamablehttp_client(server_url) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"Available tools for {server_url}: {[tool.name for tool in tools.tools]}")

weather_mcp_endpoint="https://apim-quox5nktyqj66.azure-api.net/weather-mcp/mcp"
test_url = "https://demo-ai-gateway-chris.azure-api.net/diabetes-prediction-mcp/mcp"
if __name__ == "__main__":
    asyncio.run(list_tools(test_url))


Available tools for https://apim-lqk2abkkuzazg.azure-api.net/test-prediction-mcp/mcp: ['generateAForecastPrediction']


### 🗑️ Clean up resources

When you're finished with the lab, run the [clean-up-resources notebook](clean-up-resources.ipynb) to remove the deployed resources.
